<a href="https://colab.research.google.com/github/Manojkumarfsd/Machine-Learning_Day08/blob/main/ABtesting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
np.random.seed(42)

In [11]:
#Baseline: Average shopper spends around ₹1,500 with some variation
control_spend = np.random.normal(loc=1500, scale=300, size=1000)
test_spend = np.random.normal(loc=1500, scale=300, size=1000)

In [12]:
# 3. INJECT THE WHALE OUTLIER INTO THE TEST GROUP
# One business buyer or high-net-worth user spends ₹1,50,000 completely by chance
test_spend[0] = 150000

In [13]:
df_control = pd.DataFrame({'cohort': 'Control', 'gmv': control_spend})
df_test = pd.DataFrame({'cohort': 'Test', 'gmv': test_spend})
df_experiment = pd.concat([df_control, df_test], ignore_index=True)

In [14]:
total_control_gmv = df_experiment[df_experiment['cohort'] == 'Control']['gmv'].sum()
total_test_gmv = df_experiment[df_experiment['cohort'] == 'Test']['gmv'].sum()
incremental_gmv = total_test_gmv - total_control_gmv

In [15]:
# Run standard T-test to fetch p-value
t_stat, p_val_with_whale = stats.ttest_ind(
    df_experiment[df_experiment['cohort'] == 'Test']['gmv'],
    df_experiment[df_experiment['cohort'] == 'Control']['gmv']
)

print("="*60)
print("📊 ANALYSIS WITH WHALE INCLUDED")
print("="*60)
print(f"Total Control GMV:   ₹{total_control_gmv:,.2f}")
print(f"Total Test GMV:      ₹{total_test_gmv:,.2f}")
print(f"🔥 Raw Incremental Lift: ₹{incremental_gmv:,.2f}")
print(f"P-Value:             {p_val_with_whale:.4f} (Looks significant!)")

📊 ANALYSIS WITH WHALE INCLUDED
Total Control GMV:   ₹1,501,750.26
Total Test GMV:      ₹1,643,456.58
🔥 Raw Incremental Lift: ₹141,706.31
P-Value:             0.3421 (Looks significant!)


In [16]:
threshold = df_experiment['gmv'].quantile(0.99)
df_clean = df_experiment[df_experiment['gmv'] <= threshold]

clean_control_gmv = df_clean[df_clean['cohort'] == 'Control']['gmv'].sum()
clean_test_gmv = df_clean[df_clean['cohort'] == 'Test']['gmv'].sum()
clean_incremental_gmv = clean_test_gmv - clean_control_gmv

_, p_val_clean = stats.ttest_ind(
    df_clean[df_clean['cohort'] == 'Test']['gmv'],
    df_clean[df_clean['cohort'] == 'Control']['gmv']
)

In [17]:
print("\n" + "="*60)
print("🛡️ SANITY CHECK: ANALYSIS WITH WHALES REMOVED")
print("="*60)
print(f"Clean Control GMV:   ₹{clean_control_gmv:,.2f}")
print(f"Clean Test GMV:      ₹{clean_test_gmv:,.2f}")
print(f"📉 True Behavioral Lift: ₹{clean_incremental_gmv:,.2f}")
print(f"Actual P-Value:      {p_val_clean:.4f}")

print("\n💡 CONCLUSION:")
if p_val_clean > 0.05:
    print("❌ The experiment FAILED! The apparent success was entirely driven by 1 person.")
else:
    print("✅ The experiment SUCCESS! The lift is real across the broad user base.")
print("="*60)


🛡️ SANITY CHECK: ANALYSIS WITH WHALES REMOVED
Clean Control GMV:   ₹1,476,052.60
Clean Test GMV:      ₹1,475,099.01
📉 True Behavioral Lift: ₹-953.59
Actual P-Value:      0.7618

💡 CONCLUSION:
❌ The experiment FAILED! The apparent success was entirely driven by 1 person.
